# <font color="#418FDE" size="10" uppercase>**C: SimCLR Contrastive Experiments**</font>
----

> TPU edition: 2026-09-25. Original GPU lecture: 2024-03-12.

By the end of this lecture, you will be able to:
* Create SimCLR experiments without pre-trained weights.
* Create SimCLR experiments with pre-trained weights.
* Explain why a TPU needs a distributed, compiled contrastive training step.


## Running this notebook on a Colab TPU

1. In Google Colab, choose **Runtime → Change runtime type → Hardware accelerator → TPU**. Select a v5e or v6e TPU if Colab offers one. The notebook will stop with an error if it cannot connect to a TPU; it will not silently train on a CPU.
2. Run the cells in order, starting with the installation cell. The current Colab TPU runtime does not include TensorFlow by default. We use TensorFlow TPU **2.19.1**, the version used in the [Keras TPU example](https://keras.io/keras_rs/examples/distributed_embedding_tf/). If Colab asks you to restart after installation, restart and run all cells again. For a Python compatibility error, choose **Runtime Version → 2026.07** (Python 3.12) in the same runtime dialog, then rerun the notebook.
3. The TPU connection cell runs once for both experiments. **Keep the runtime connected** between the two sections. You can lower the epoch values in either hyperparameter cell for a quick trial; the original full epoch settings remain the defaults.

**What changes on a TPU?** A GPU notebook can update a model one Python batch at a time. A TPU needs the training calculation compiled into a TensorFlow graph. `TPUStrategy` places copies of the model on the available TPU cores, sends each core a portion of the batch, and combines their weight updates. Keras `fit()` handles this for the labeled models. We explicitly distribute and compile the custom SimCLR training step. The two random image views are made by the input pipeline before the batch reaches the TPU, so image rotation and cropping remain straightforward.

This is still the same lesson: use images without labels to learn a representation, then train a classifier with limited labels. See the [TensorFlow TPU guide](https://www.tensorflow.org/guide/tpu) and the [custom distributed training guide](https://www.tensorflow.org/tutorials/distribute/custom_training) for background.


In [1]:
#@title Install TensorFlow for the Colab TPU (run once, before importing TensorFlow)
import sys

# The TensorFlow TPU 2.19.1 package has a wheel for Colab's Python 3.12.
if sys.version_info >= (3, 13):
    raise RuntimeError("Choose Colab runtime version 2026.07 (Python 3.12), then rerun this cell.")

# The TPU package includes the TensorFlow code and its TPU runtime library.
%pip -q install tensorflow-tpu==2.19.1 --find-links=https://storage.googleapis.com/libtpu-tf-releases/index.html


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.4/255.4 MB 3.2 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 166.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 184.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 180.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 35.7 MB/s eta 0:00:00


In [2]:
#@title Connect to the TPU (run once for both experiments)
import os
import libtpu

# Set these environment variables BEFORE importing TensorFlow. They follow
# Keras's TensorFlow TPU example for the PJRT device and compiler bridge.
os.environ['PJRT_DEVICE'] = 'TPU'
os.environ['NEXT_PLUGGABLE_DEVICE_USE_C_API'] = 'true'
os.environ['TF_PLUGGABLE_DEVICE_LIBRARY_PATH'] = libtpu.get_library_path()
os.environ['TF_XLA_FLAGS'] = (
    '--tf_mlir_enable_mlir_bridge=true '
    '--tf_mlir_enable_convert_control_to_data_outputs_pass=true '
    '--tf_mlir_enable_merge_control_flow_pass=true'
)
os.environ['KERAS_BACKEND'] = 'tensorflow'

import tensorflow as tf

# Colab's newer TPU runtimes expose the accelerator as a local TPU.
# If you chose a CPU or GPU runtime, these lines stop rather than using it.
resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)

print('TensorFlow version:', tf.__version__)
print('TPU devices:', tf.config.list_logical_devices('TPU'))
print('Number of TPU replicas:', strategy.num_replicas_in_sync)
assert tf.config.list_logical_devices('TPU'), 'No TPU detected. Select a TPU runtime in Colab.'


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


TensorFlow version: 2.19.1
TPU devices: [LogicalDevice(name='/device:TPU:0', device_type='TPU')]
Number of TPU replicas: 1


### How to read the two training paths

**FSP** (*fully supervised*): DenseNet sees only labeled images and learns to predict one of the ten CIFAR-10 classes. **PRX** (*pretext*): DenseNet sees all 50,000 training images without their labels. Each image is changed twice, like taking two slightly different camera views of the same machine part. SimCLR rewards the model when it recognizes the two views as related and distinguishes them from views of other images. **DWM** (*downstream*): a new classifier reuses the features learned in PRX and learns the same ten classes from the limited labeled data.

Exactly **1,000 distinct images** provide labels. We use 896 to fit FSP and DWM and 104 for both models' early stopping, so the **10,000-image test set is used only for final evaluation**. These counts make complete batches on the TPU. PRX uses all training images without labels, including the 104 validation images. Both supervised paths use the same labeled split and start their final classifier from the same initial weights. They need not produce identical accuracies: their image features were learned differently.

For the TPU, `batch_prx` is the number of *original* images **per core** times the number of cores. There are twice as many augmented views. Each core compares the pairs assigned to it. Increasing `batch_prx` uses more memory but gives SimCLR more other images to compare; see [TensorFlow's explanation of distributed datasets](https://www.tensorflow.org/guide/distributed_training#use_tfdistributestrategy_with_keras_modelfit). The final incomplete PRX batch is dropped to give the TPU a fixed batch shape. All 896 labeled fitting images fit in complete batches of 64; the 104 validation images also divide evenly among the usual one to eight TPU cores.


## **1. SimCLR Experiment - without Pre-Trained Weights**

> In this experiment, the focus is on training models <u>***without***</u> the advantage of pre-trained parameters due to the unavailability of a network trained on a similar data distribution and using the SimCLR technique. Besides, we assume we have limited labeled data from the CIFAR-10 dataset, specifically using only 1000 labeled images. This way, we simulate a real-world labeling challenge scenario.

> The approach involves developing a SimCLR unsupervised contrastive pretext model (`model_prx`) utilizing all training inputs, which is then transfer-learned and fine-tuned on the 1000 labeled images for the downstream task (`model_dwm`). This fine-tuned model is used to label testing images.

> For comparative analysis, a fully supervised model (`model_fsp`) is also trained solely on the 1000 labeled images. The key comparison is between the accuracies of `model_fsp` and `model_dwm` on the testing data, highlighting the effectiveness of the contrastive pretext strategy with limited labeled data.

In [3]:
#@title Imports & Hyper-Parameters
'''
Runtime: Google Colab TPU v5e or v6e. Run the two TPU setup cells first.
The original epoch counts are preserved; runtime depends on TPU availability.

Abbreviations:
    acc: accuracy
    datain: input data
    dataou: output data
    fsp: fully supervised learning
    prx: pretext
    dwm: downstream
    trf: transfer learning
    fnt: fine-tuning
    lr: learning rate
    tf: tensorflow
    tr: training
    te: testing

Remember: we want to practice Self-Supervised Learning (prx + dwm) and compare
it with fsp scenario. So, we take some measures to make sure we have a fair comparison.
'''

# Keep the TPU strategy initialized above. Clear any previous Keras state.
import gc
import pandas as pd
from tqdm import tqdm
import time

tf.keras.backend.clear_session()
gc.collect()

# Use the same random seed for the labeled subset and initial weights.
# Section 2 resets the seed to use the same 1,000 labeled examples.
tf.keras.utils.set_random_seed(42)

'''
Hyper-parameters
'''

num_labeled  = 1000

## learning rates
lr_fsp     = 0.001
lr_prx     = 0.001
lr_dwm_trf = 0.01
lr_dwm_fnt = 0.001

## batch sizes
batch_fsp = 64
# Keep 32 original images (64 augmented views) on EACH TPU replica.
# The batch is automatically split if Colab gives us multiple TPU cores.
batch_prx = 32 * strategy.num_replicas_in_sync
batch_dwm = 64

## epochs: We keep epoch_fsp  = epoch_dwm_trf + epoch_dwm_fnt
## for a fair comparison.
epoch_fsp     = 25
epoch_prx     = 25
epoch_dwm_trf = 15
epoch_dwm_fnt = 10

## Image resolution for upscaling CIFAR10 data
global res
res = 128

# Keras handles these labeled batches; the SimCLR batch is divided by replica count.
print('Original images in one SimCLR batch:', batch_prx)
print('Augmented images in one SimCLR batch:', 2 * batch_prx)

Original images in one SimCLR batch: 32
Augmented images in one SimCLR batch: 64


In [4]:
#@title Data Preparation


# Load CIFAR-10 dataset, splitting into training and testing sets
!mkdir -p ~/.keras/datasets/
!if [ ! -f ~/.keras/datasets/cifar-10-batches-py-target_archive ]; then \
    wget -q -O ~/.keras/datasets/cifar-10-batches-py-target_archive \
    https://storage.googleapis.com/535743/datasets/cifar-10-python.tar.gz; \
fi

(datain_tr, dataou_tr), (datain_te, dataou_te) = tf.keras.datasets.cifar10.load_data()

# Normalize training and testing input data from integers (0-255) to floats (0-1)
datain_tr = datain_tr.astype('float32') / 255.0
datain_te = datain_te.astype('float32') / 255.0

# Convert training and testing output labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr)
dataou_te = tf.keras.utils.to_categorical(dataou_te)

# Print the shapes of the input and output data sets for both training and testing
print('Shape of datain_tr: {}'.format(datain_tr.shape))
print('Shape of datain_te: {}'.format(datain_te.shape))
print('Shape of dataou_tr: {}'.format(dataou_tr.shape))
print('Shape of dataou_te: {}'.format(dataou_te.shape))

# Data Augmentation

# Define two data augmentations.
fun_augment_01 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

fun_augment_02 = tf.keras.Sequential([
    tf.keras.layers.RandomCrop(height = int(res/2), width = int(res/2)),
    tf.keras.layers.Resizing(height = res, width  = res)
])


# We only have input data for prx; we don't use any prx validation data in this experiment
datain_tr_prx = datain_tr  # Reuse the images; no extra large copy is needed.

# Limit the labeled training data

# Select 1,000 DIFFERENT labeled images. randint could select one image twice.
index_tr = tf.random.shuffle(tf.range(datain_tr.shape[0]), seed=42)[:num_labeled]

# Gather the selected subset of labeled training data
datain_tr_labeled = tf.gather(datain_tr, index_tr, axis=0)
dataou_tr_labeled = tf.gather(dataou_tr, index_tr, axis=0)

# Keep 104 of the 1,000 labeled images for validation (early stopping).
# Both labeled models train on the SAME remaining 896 labeled images.
# 896 is 14 complete batches of 64; 104 splits evenly across 1, 2, 4 or 8 cores.
# All 50,000 training images, without their labels, are available to SimCLR.
num_val = 104
datain_vl = datain_tr_labeled[:num_val]
dataou_vl = dataou_tr_labeled[:num_val]
datain_tr_fsp = datain_tr_labeled[num_val:]
dataou_tr_fsp = dataou_tr_labeled[num_val:]
datain_tr_dwm = datain_tr_fsp
dataou_tr_dwm = dataou_tr_fsp

# Define an upscaling function to be later used in the tf dataset API
def fun_upscale(image, label=None):
    # The original CIFAR-10 images are 32 x 32. DenseNet receives res x res.
    image_resized = tf.image.resize_with_pad(image, res, res,
                                             method=tf.image.ResizeMethod.BILINEAR)
    if label is None:
        return image_resized  # Unlabeled images for SimCLR.
    return image_resized, label

# The first augmentation flips/rotates the images. The second crops/resizes.
# Create TWO different views of each image and place them next to each other:
# view 1 of image 0, view 2 of image 0, view 1 of image 1, view 2 of image 1...
# SimCLR needs this exact ordering to know which two images are a positive pair.
def fun_two_views(batch_in):
    x_tilda_01 = fun_augment_01(batch_in, training=True)
    x_tilda_02 = fun_augment_02(batch_in, training=True)
    paired_views = tf.stack([x_tilda_01, x_tilda_02], axis=1)
    return tf.reshape(paired_views, (-1, res, res, 3))

# tf.data makes the augmented views on the host BEFORE sending them to a TPU.
# Dropping the final incomplete SimCLR batch keeps each TPU batch shape fixed.
buffer_size = datain_tr_fsp.shape[0]
dataset_tr_fsp = (tf.data.Dataset.from_tensor_slices((datain_tr_fsp, dataou_tr_fsp))
                  .shuffle(buffer_size, reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_fsp, drop_remainder=True)
                  .prefetch(tf.data.AUTOTUNE))

dataset_tr_prx = (tf.data.Dataset.from_tensor_slices(datain_tr_prx)
                  # Shuffle small 32 x 32 images BEFORE enlarging them.
                  # Shuffling all 50,000 after resizing could exhaust host RAM.
                  .shuffle(datain_tr_prx.shape[0], reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_prx, drop_remainder=True)
                  .map(fun_two_views, num_parallel_calls=tf.data.AUTOTUNE)
                  .prefetch(tf.data.AUTOTUNE))

dataset_tr_dwm = (tf.data.Dataset.from_tensor_slices((datain_tr_dwm, dataou_tr_dwm))
                  .shuffle(buffer_size, reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_dwm, drop_remainder=True)
                  .prefetch(tf.data.AUTOTUNE))

dataset_vl = (tf.data.Dataset.from_tensor_slices((datain_vl, dataou_vl))
              .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(128).prefetch(tf.data.AUTOTUNE))

dataset_te = (tf.data.Dataset.from_tensor_slices((datain_te, dataou_te))
              .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(128).prefetch(tf.data.AUTOTUNE))

# The custom SimCLR training loop needs explicit dataset distribution.
# Keras model.fit() distributes the labeled datasets for us automatically.
distributed_dataset_tr_prx = strategy.experimental_distribute_dataset(dataset_tr_prx)
steps_prx = datain_tr_prx.shape[0] // batch_prx
print('SimCLR batches per epoch:', steps_prx)
print('Unlabeled images omitted in each epoch:', datain_tr_prx.shape[0] % batch_prx)


Shape of datain_tr: (50000, 32, 32, 3)
Shape of datain_te: (10000, 32, 32, 3)
Shape of dataou_tr: (50000, 10)
Shape of dataou_te: (10000, 10)
SimCLR batches per epoch: 1562
Unlabeled images omitted in each epoch: 16


In [5]:
#@title Create Models FSP and PRX

with strategy.scope():
    # Build DenseNet121 and both models inside the TPU strategy.
    model_base = tf.keras.applications.DenseNet121(include_top=False,
                                                   weights=None,
                                                   input_shape=(res, res, 3))

    # We clone model_base to create model_base_fsp and model_base_prx model.
    # P.S. We will clone the model_dws after the pretext task using model_prx.
    model_base_fsp = tf.keras.models.clone_model(model_base)
    model_base_prx = tf.keras.models.clone_model(model_base)

    # We set the parameters of model_base_fsp and model_base_prx to be the same as the
    # randomly generated parameters of model_base to have both models' initial
    # weights (i.e., start-point) the same for a fair comparison.
    model_base_fsp.set_weights(model_base.get_weights())
    model_base_prx.set_weights(model_base.get_weights())

    print('Example of weights in the 3rd  layer of model_base_fsp:', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    print('Example of weights in the 3rd  layer of model_base_prx:', model_base_prx.layers[2].weights[0][0][0][0][:2])

    # SimCLR projector
    projector_input_shape = int(tf.math.reduce_prod(model_base_prx.output_shape[1:]))
    model_projector       = tf.keras.Sequential([tf.keras.Input(shape=(projector_input_shape,)),
                                                 tf.keras.layers.Dense(512, activation = 'relu'),
                                                 tf.keras.layers.Dense(128, activation = 'relu')])


    # Now we create the model_fsp and model_prx.
    inputs_fsp  = tf.keras.Input(shape=(res, res, 3))
    x_fsp       = model_base_fsp(inputs_fsp)
    x_fsp       = tf.keras.layers.Flatten()(x_fsp)
    x_fsp       = tf.keras.layers.BatchNormalization()(x_fsp)
    outputs_fsp = tf.keras.layers.Dense(10, activation='softmax')(x_fsp) # ten outputs
    model_fsp   = tf.keras.Model(inputs_fsp, outputs_fsp)


    inputs_prx  = tf.keras.Input(shape=(res, res, 3))
    x_prx       = model_base_prx(inputs_prx)
    x_prx       = tf.keras.layers.Flatten()(x_prx)
    x_prx       = tf.keras.layers.BatchNormalization()(x_prx)
    outputs_prx = model_projector(x_prx) # there are no actual outputs but the last layer of projector
    model_prx   = tf.keras.Model(inputs_prx, outputs_prx)


Example of weights in the 3rd  layer of model_base_fsp: tf.Tensor([0.0368682  0.01207029], shape=(2,), dtype=float32)
Example of weights in the 3rd  layer of model_base_prx: tf.Tensor([0.0368682  0.01207029], shape=(2,), dtype=float32)


In [6]:
#@title Some Useful Functions
'''
Abbreviations:
    datain: Input data
    tf    : TensorFlow
'''

# Function definition for training a model using the SimCLR approach
def fun_train_simclr(model, dataset, epochs=100, verbose=1,
                     patience=3, learning_rate=0.001):
    # Create the optimizer under the same strategy as the DenseNet model.
    with strategy.scope():
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        optimizer.build(model.trainable_variables)

    # One TPU core receives a portion of the batch. Each portion already has
    # two adjacent views of every image. TensorFlow compiles this whole step.
    @tf.function
    def distributed_train_step(batch_in):
        def train_step(images):
            with tf.GradientTape() as tape:
                z_estimate = model(images, training=True)
                loss_batch = fun_simclr_loss(z_estimate)
                # After the TPU cores combine their gradients, this gives the
                # average loss across ALL cores, rather than their sum.
                loss_batch = loss_batch / strategy.num_replicas_in_sync
            gradients = tape.gradient(loss_batch, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            return loss_batch

        losses_per_replica = strategy.run(train_step, args=(batch_in,))
        return strategy.reduce(tf.distribute.ReduceOp.SUM,
                               losses_per_replica, axis=None)

    loss = []
    for epoch in range(epochs):
        loss_running = 0.0
        pbar = tqdm(dataset, total=steps_prx,
                    desc=f'SimCLR Training: {epoch + 1:03d}/{epochs}',
                    ncols=125, leave=True)

        for batch_num, batch in enumerate(pbar, 1):
            loss_batch = distributed_train_step(batch)
            loss_running += float(loss_batch.numpy())
            if verbose:
                pbar.set_postfix(loss=f'{loss_running / batch_num:.6f}')

        loss.append(loss_running / batch_num)
        if epoch >= patience and loss[-1] > min(loss[:-patience]):
            print('Early stopping due to no improvement in loss.')
            break

    return model, loss


# SimCLR's contrastive loss, also called NT-Xent.
@tf.function
def fun_simclr_loss(z_estimate):
    # z_estimate contains the learned representation of each image view.
    # Converting to float32 keeps the similarities numerically stable.
    z_estimate = tf.math.l2_normalize(tf.cast(z_estimate, tf.float32), axis=1)
    num = tf.shape(z_estimate)[0]  # Twice the number of original images on this core.
    temperature = 0.1

    # One score for every possible pair in this core's portion of the batch.
    similarity = tf.matmul(z_estimate, z_estimate, transpose_b=True)
    logits = similarity / temperature

    # An image should NOT be matched with itself. Give those scores a very
    # negative value so softmax ignores them in its comparisons.
    logits = logits - tf.eye(num, dtype=tf.float32) * 1e9

    # The views are interleaved: 0 pairs with 1, 2 pairs with 3, and so on.
    indices = tf.range(num)
    positive = tf.where(indices % 2 == 0, indices + 1, indices - 1)

    # For each view, reward its matching augmented view and compare it with
    # every other image view on this core. No class labels are involved.
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=positive, logits=logits)
    return tf.reduce_mean(losses)

# Function to compile a model with specified learning rate
def fun_model_compile(model, learning_rate, loss = 'categorical_crossentropy', metrics = ['accuracy']):
    # Compiles the model with Adam optimizer, categorical crossentropy as loss, and tracks accuracy
    with strategy.scope():
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                      loss=loss,
                      metrics=metrics)
    return model

# Function to print the summary of the model
def fun_model_summary(model):
    model.summary()  # Prints the summary of the model
    print('\n')  # Prints a newline for readability

# Function to define callbacks for training
def fun_model_callbacks():
    # Early stopping callback to stop training when val_loss doesn't improve
    cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    # Reduce learning rate callback when val_loss plateaus
    cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)
    return [cb_early_stopping, cb_reduce_lr]

# Function to train the models other than SimCLR with given dataset, epochs, and callbacks
def fun_model_train(model, dataset_tr, dataset_vl, epochs, callbacks):
    model.fit(dataset_tr,  # Training data
              epochs          = epochs,  # Number of epochs to train for
              verbose         = 1,  # Verbosity mode
              shuffle         = True,  # Whether to shuffle the training data
              validation_data = dataset_vl,  # Validation data
              callbacks       = callbacks)  # Callbacks for training
    return model

# Function to evaluate the model with a given dataset
def fun_model_evaluate(model, data):
    _, acc = model.evaluate(data, verbose=0)
    return [acc]

# Function to compare the weights of base and trained models in fsp and prx tasks
def fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx):
    print('\n')
    # Print examples of weights from the 3rd layer of fully supervised base model
    print('Example of weights in the 3rd  layer of model_base_fsp: ', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the 3rd layer of pretext base model
    print('Example of weights in the 3rd  layer of model_base_prx: ', model_base_prx.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the last layer of fully supervised trained model
    print('Example of weights in the last layer of model_fsp     : ', model_fsp.layers[-1].weights[0][:1][0][:2])
    # Print examples of weights from the last layer of pretext trained model
    print('Example of weights in the last layer of model_prx     : ', model_prx.layers[-1].weights[0][:1][0][:2])


In [7]:
#@title Training Models
# Capture the start time
t0 = time.time()

# Initiate a result dictionary
results = {}

'''
Training model_fsp.
'''

print('\nTraining model_fsp\n')

# No pre-trained weights, no freezing!
model_base_fsp.trainable = True

# No pre-trained weights, no freezing!
model_fsp.layers[-2].trainable = True

# Compile the model with specified learning rate for transfer learning phase.
model_fsp = fun_model_compile(model_fsp, lr_fsp)

# Store the initial parameters of the last layer of model_fsp for comparison later.
outputs_fsp_initial_parameters = model_fsp.layers[-1].get_weights()

# Display the model's structure.
fun_model_summary(model_fsp)

# Prepare callbacks for early stopping and learning rate adjustment.
callbacks = fun_model_callbacks()

# Start training the model with specified datasets and number of epochs.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_vl, epoch_fsp, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_trf'] = fun_model_evaluate(model_fsp, dataset_te)

# Check and display changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Training model_prx.
'''

print('\nTraining model_prx\n')

# No pre-trained weights, no freezing!
model_base_prx.trainable = True

# No pre-trained weights, no freezing!
model_prx.layers[-2].trainable = True

# Show the model's structure.
fun_model_summary(model_prx)

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                distributed_dataset_tr_prx,
                                epochs     = epoch_prx,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)


'''
Construct and prepare model_dwm for transfer learning.
'''
# Constructing model_dwm utilizing the pre-trained model_base_prx, incorporating
# a dense layer similar to that in model_fsp.
with strategy.scope():
    inputs_dwm = tf.keras.Input(shape=(res, res, 3))
    x_dwm = model_base_prx(inputs_dwm)  # Use the trained base model.
    x_dwm = tf.keras.layers.Flatten()(x_dwm)
    x_dwm = tf.keras.layers.BatchNormalization()(x_dwm)
    outputs_dwm = tf.keras.layers.Dense(10, activation='softmax')(x_dwm)  # Set for 10 output classes.
    model_dwm = tf.keras.Model(inputs_dwm, outputs_dwm)


# For equitable comparison between fsp and dwm, we align the parameters of dwm's
# final dense layer with outputs_fsp_initial_parameters.
# Additionally, we transfer and freeze the pre-trained weights of the preceding
# dense layer from prx into dwm's structure to leverage their learned
# representations.
# Set the initial parameters for fair comparison and incorporate pre-trained layers.
model_dwm.layers[-1].set_weights(outputs_fsp_initial_parameters)  # Match initial output layer parameters.
model_dwm.layers[-2].set_weights(model_prx.layers[-2].get_weights())  # Use trained batch-norm weights.

'''
Transfer learning for model_dwm.
'''

print('\nInitializing transfer learning for model_dwm\n')

# Freeze base and selected layers for the initial phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_dwm.layers[-2].trainable = False

# Compile the model for this phase.
model_dwm = fun_model_compile(model_dwm, lr_dwm_trf)

# Display the model structure.
fun_model_summary(model_dwm)

# Set callbacks for training.
callbacks = fun_model_callbacks()

# Begin training the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_vl, epoch_dwm_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_trf'] = fun_model_evaluate(model_dwm, dataset_te)

# Check parameter changes after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Fine-tuning for model_dwm.
'''

print('\nStarting fine-tuning for model_dwm\n')

# Unfreeze layers for fine-tuning.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_dwm.layers[-2].trainable = True

# Recompile with adjusted learning rate for fine-tuning.
model_dwm = fun_model_compile(model_dwm, lr_dwm_fnt)

# Display the model's updated structure.
fun_model_summary(model_dwm)

# Prepare callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed to fine-tune the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_vl, epoch_dwm_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_fnt'] = fun_model_evaluate(model_dwm, dataset_te)

# Evaluate parameter changes post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)

# Display the results.
print('---- Accuracies ----')
print(pd.DataFrame(results).head())
print('\n')
print("Time duration of experiment (sec): ", time.time() - t0)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
# from google.colab import runtime
# import time
# time.sleep(10)
# runtime.unassign()



Training model_fsp



Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16384)          │        65,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │       163,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,266,890 (27.72 MB)

 Trainable params: 7,150,474 (27.28 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16384)          │        65,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │       163,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,266,890 (27.72 MB)

 Trainable params: 7,150,474 (27.28 MB)

 Non-trainable params: 116,416 (454.75 KB)



Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.8650 - loss: 1.5183 - val_accuracy: 0.3846 - val_loss: 15.2654 - learning_rate: 0.0010
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9074 - loss: 0.6957 - val_accuracy: 0.3846 - val_loss: 14.6020 - learning_rate: 0.0010
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9542 - loss: 0.2570 - val_accuracy: 0.3750 - val_loss: 15.8852 - learning_rate: 0.0010
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.9565 - loss: 0.2558 - val_accuracy: 0.3654 - val_loss: 14.0784 - learning_rate: 0.0010
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.9621 - loss: 0.2393 - val_accuracy: 0.3269 - val_loss: 15.0553 - learning_rate: 0.0010
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.9710 - loss: 0.2008 - val_accuracy: 0.3462 - val_loss: 16.0349 - learning_rate: 0.0010
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9732 - loss: 0.1500 -

### What changes when the base model starts with ImageNet weights?

This section repeats the same FSP versus PRX plus DWM comparison, but both base DenseNets initially have the **same ImageNet weights**. During the first phase, we freeze the base model and train only the added head. During fine-tuning, we allow the base weights to change. The second experiment applies [DenseNet's required `preprocess_input`](https://www.tensorflow.org/api_docs/python/tf/keras/applications/densenet/preprocess_input) to every image before feeding it to the ImageNet model.

All code cells here run after Section 1 in one Colab session. The setup below frees the old models and keeps the TPU connected. The two sections each repeat their helper functions so students can follow the original notebook's order.


## **2. SimCLR Experiment - with Pre-Trained Weights**

> In this experiment, the focus is on training models <u>***with***</u> the advantage of pre-trained parameters due to the availability of a network trained on a similar data distribution and using the SimCLR technique. Besides, we assume we have limited labeled data from the CIFAR-10 dataset, specifically using only 1000 labeled images. This way, we simulate a real-world labeling challenge scenario.

> The approach involves developing a SimCLR unsupervised contrastive pretext model (`model_prx`) utilizing all training inputs, which is then transfer-learned and fine-tuned on the 1000 labeled images for the downstream task (`model_dwm`). This fine-tuned model is used to label testing images.

> For comparative analysis, a fully supervised model (`model_fsp`) is also trained solely on the 1000 labeled images. The key comparison is between the accuracies of `model_fsp` and `model_dwm` on the testing data, highlighting the effectiveness of the contrastive pretext strategy with limited labeled data.

In [8]:
#@title Imports & Hyper-Parameters
'''
Runtime: Google Colab TPU v5e or v6e (same connected TPU as Section 1).

Abbreviations:
    acc: accuracy
    datain: input data
    dataou: output data
    fsp: fully supervised learning
    prx: pretext
    dwm: downstream
    trf: transfer learning
    fnt: fine-tuning
    lr: learning rate
    tf: tensorflow
    tr: training
    te: testing

Remember: we want to practice Self-Supervised Learning (prx + dwm) and compare
it with fsp scenario. So, we take some measures to make sure we have a fair comparison.
'''

# After Section 1, release its models before loading ImageNet weights.
# Keep the already initialized TPU strategy; reinitializing can invalidate it.
del model_base, model_base_fsp, model_base_prx, model_projector
del model_fsp, model_prx, model_dwm
del dataset_tr_fsp, dataset_tr_prx, dataset_tr_dwm, dataset_vl, dataset_te
del distributed_dataset_tr_prx
del datain_tr, datain_te, datain_tr_prx
tf.keras.backend.clear_session()

import gc
gc.collect()
import pandas as pd
from tqdm import tqdm
import time

# Reproduce the same labeled subset selected in Section 1.
tf.keras.utils.set_random_seed(42)

'''
Hyper-parameters
'''

num_labeled  = 1000

## learning rates
lr_fsp_trf = 0.01
lr_fsp_fnt = 0.0001
lr_prx_trf = 0.01
lr_prx_fnt = 0.00001
lr_dwm_trf = 0.01
lr_dwm_fnt = 0.0001

## batch sizes
batch_fsp = 64
# Preserve 64 original images per core for the contrastive loss.
batch_prx = 64 * strategy.num_replicas_in_sync
batch_dwm = 64

## epochs: We keep epoch_fsp_trf + epoch_fsp_fnt = epoch_dwm_trf + epoch_dwm_fnt
## for a fair comparison.
epoch_fsp_trf = 15
epoch_fsp_fnt = 10
epoch_prx_trf = 15
epoch_prx_fnt = 10
epoch_dwm_trf = 15
epoch_dwm_fnt = 10

## Image resolution for upscaling CIFAR10 data
global res
res = 128

print('Original images in one SimCLR batch:', batch_prx)
print('Augmented images in one SimCLR batch:', 2 * batch_prx)

Original images in one SimCLR batch: 64
Augmented images in one SimCLR batch: 128


In [9]:
#@title Data Preparation

# Load CIFAR-10 dataset, splitting into training and testing sets
(datain_tr, dataou_tr), (datain_te, dataou_te) = tf.keras.datasets.cifar10.load_data()

# Normalize training and testing input data from integers (0-255) to floats (0-1)
datain_tr = datain_tr.astype('float32') / 255.0
datain_te = datain_te.astype('float32') / 255.0

# Convert training and testing output labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr)
dataou_te = tf.keras.utils.to_categorical(dataou_te)

# Print the shapes of the input and output data sets for both training and testing
print('Shape of datain_tr: {}'.format(datain_tr.shape))
print('Shape of datain_te: {}'.format(datain_te.shape))
print('Shape of dataou_tr: {}'.format(dataou_tr.shape))
print('Shape of dataou_te: {}'.format(dataou_te.shape))

# Data Augmentation

# Define two data augmentations.
fun_augment_01 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

fun_augment_02 = tf.keras.Sequential([
    tf.keras.layers.RandomCrop(height = int(res/2), width = int(res/2)),
    tf.keras.layers.Resizing(height = res, width  = res)
])


# We only have input data for prx; we don't use any prx validation data in this experiment
datain_tr_prx = datain_tr  # Reuse the images; no extra large copy is needed.

# Limit the labeled training data

# Select 1,000 DIFFERENT labeled images. randint could select one image twice.
index_tr = tf.random.shuffle(tf.range(datain_tr.shape[0]), seed=42)[:num_labeled]

# Gather the selected subset of labeled training data
datain_tr_labeled = tf.gather(datain_tr, index_tr, axis=0)
dataou_tr_labeled = tf.gather(dataou_tr, index_tr, axis=0)

# Keep 104 of the 1,000 labeled images for validation (early stopping).
# Both labeled models train on the SAME remaining 896 labeled images.
# 896 is 14 complete batches of 64; 104 splits evenly across 1, 2, 4 or 8 cores.
# All 50,000 training images, without their labels, are available to SimCLR.
num_val = 104
datain_vl = datain_tr_labeled[:num_val]
dataou_vl = dataou_tr_labeled[:num_val]
datain_tr_fsp = datain_tr_labeled[num_val:]
dataou_tr_fsp = dataou_tr_labeled[num_val:]
datain_tr_dwm = datain_tr_fsp
dataou_tr_dwm = dataou_tr_fsp

# Define an upscaling function to be later used in the tf dataset API
def fun_upscale(image, label=None):
    # The original CIFAR-10 images are 32 x 32. DenseNet receives res x res.
    image_resized = tf.image.resize_with_pad(image, res, res,
                                             method=tf.image.ResizeMethod.BILINEAR)
    # ImageNet DenseNet expects its own color-channel normalization.
    # Undo our [0, 1] scaling before applying that standard preprocessing.
    image_resized = tf.keras.applications.densenet.preprocess_input(image_resized * 255.0)
    if label is None:
        return image_resized  # Unlabeled images for SimCLR.
    return image_resized, label

# The first augmentation flips/rotates the images. The second crops/resizes.
# Create TWO different views of each image and place them next to each other:
# view 1 of image 0, view 2 of image 0, view 1 of image 1, view 2 of image 1...
# SimCLR needs this exact ordering to know which two images are a positive pair.
def fun_two_views(batch_in):
    x_tilda_01 = fun_augment_01(batch_in, training=True)
    x_tilda_02 = fun_augment_02(batch_in, training=True)
    paired_views = tf.stack([x_tilda_01, x_tilda_02], axis=1)
    return tf.reshape(paired_views, (-1, res, res, 3))

# tf.data makes the augmented views on the host BEFORE sending them to a TPU.
# Dropping the final incomplete SimCLR batch keeps each TPU batch shape fixed.
buffer_size = datain_tr_fsp.shape[0]
dataset_tr_fsp = (tf.data.Dataset.from_tensor_slices((datain_tr_fsp, dataou_tr_fsp))
                  .shuffle(buffer_size, reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_fsp, drop_remainder=True)
                  .prefetch(tf.data.AUTOTUNE))

dataset_tr_prx = (tf.data.Dataset.from_tensor_slices(datain_tr_prx)
                  # Shuffle small 32 x 32 images BEFORE enlarging them.
                  # Shuffling all 50,000 after resizing could exhaust host RAM.
                  .shuffle(datain_tr_prx.shape[0], reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_prx, drop_remainder=True)
                  .map(fun_two_views, num_parallel_calls=tf.data.AUTOTUNE)
                  .prefetch(tf.data.AUTOTUNE))

dataset_tr_dwm = (tf.data.Dataset.from_tensor_slices((datain_tr_dwm, dataou_tr_dwm))
                  .shuffle(buffer_size, reshuffle_each_iteration=True)
                  .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(batch_dwm, drop_remainder=True)
                  .prefetch(tf.data.AUTOTUNE))

dataset_vl = (tf.data.Dataset.from_tensor_slices((datain_vl, dataou_vl))
              .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(128).prefetch(tf.data.AUTOTUNE))

dataset_te = (tf.data.Dataset.from_tensor_slices((datain_te, dataou_te))
              .map(fun_upscale, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(128).prefetch(tf.data.AUTOTUNE))

# The custom SimCLR training loop needs explicit dataset distribution.
# Keras model.fit() distributes the labeled datasets for us automatically.
distributed_dataset_tr_prx = strategy.experimental_distribute_dataset(dataset_tr_prx)
steps_prx = datain_tr_prx.shape[0] // batch_prx
print('SimCLR batches per epoch:', steps_prx)
print('Unlabeled images omitted in each epoch:', datain_tr_prx.shape[0] % batch_prx)


Shape of datain_tr: (50000, 32, 32, 3)
Shape of datain_te: (10000, 32, 32, 3)
Shape of dataou_tr: (50000, 10)
Shape of dataou_te: (10000, 10)
SimCLR batches per epoch: 781
Unlabeled images omitted in each epoch: 16


In [10]:
#@title Create Models FSP and PRX

with strategy.scope():
    # Build DenseNet121 and both models inside the TPU strategy.
    model_base = tf.keras.applications.DenseNet121(include_top=False,
                                                   weights='imagenet',
                                                   input_shape=(res, res, 3))

    # We clone model_base to create model_base_fsp and model_base_prx model.
    # P.S. We will clone the model_dws after the pretext task using model_prx.
    model_base_fsp = tf.keras.models.clone_model(model_base)
    model_base_prx = tf.keras.models.clone_model(model_base)

    # We set the parameters of model_base_fsp and model_base_prx to be the same as the
    # ImageNet parameters of model_base to have both models' initial
    # weights (i.e., start-point) the same for a fair comparison.
    model_base_fsp.set_weights(model_base.get_weights())
    model_base_prx.set_weights(model_base.get_weights())

    print('Example of weights in the 3rd  layer of model_base_fsp:', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    print('Example of weights in the 3rd  layer of model_base_prx:', model_base_prx.layers[2].weights[0][0][0][0][:2])

    # SimCLR projector
    projector_input_shape = int(tf.math.reduce_prod(model_base_prx.output_shape[1:]))
    model_projector       = tf.keras.Sequential([tf.keras.Input(shape=(projector_input_shape,)),
                                                 tf.keras.layers.Dense(512, activation = 'relu'),
                                                 tf.keras.layers.Dense(128, activation = 'relu')])


    # Now we create the model_fsp and model_prx.
    inputs_fsp  = tf.keras.Input(shape=(res, res, 3))
    x_fsp       = model_base_fsp(inputs_fsp)
    x_fsp       = tf.keras.layers.Flatten()(x_fsp)
    x_fsp       = tf.keras.layers.BatchNormalization()(x_fsp)
    outputs_fsp = tf.keras.layers.Dense(10, activation='softmax')(x_fsp) # ten outputs
    model_fsp   = tf.keras.Model(inputs_fsp, outputs_fsp)


    inputs_prx  = tf.keras.Input(shape=(res, res, 3))
    x_prx       = model_base_prx(inputs_prx)
    x_prx       = tf.keras.layers.Flatten()(x_prx)
    x_prx       = tf.keras.layers.BatchNormalization()(x_prx)
    outputs_prx = model_projector(x_prx) # there are no actual outputs but the last layer of projector
    model_prx   = tf.keras.Model(inputs_prx, outputs_prx)


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Example of weights in the 3rd  layer of model_base_fsp: tf.Tensor([0.07827556 0.01419056], shape=(2,), dtype=float32)
Example of weights in the 3rd  layer of model_base_prx: tf.Tensor([0.07827556 0.01419056], shape=(2,), dtype=float32)


In [11]:
#@title Some Useful Functions
'''
Abbreviations:
    datain: Input data
    tf    : TensorFlow
'''

# Function definition for training a model using the SimCLR approach
def fun_train_simclr(model, dataset, epochs=100, verbose=1,
                     patience=3, learning_rate=0.001):
    # Create the optimizer under the same strategy as the DenseNet model.
    with strategy.scope():
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        optimizer.build(model.trainable_variables)

    # One TPU core receives a portion of the batch. Each portion already has
    # two adjacent views of every image. TensorFlow compiles this whole step.
    @tf.function
    def distributed_train_step(batch_in):
        def train_step(images):
            with tf.GradientTape() as tape:
                z_estimate = model(images, training=True)
                loss_batch = fun_simclr_loss(z_estimate)
                # After the TPU cores combine their gradients, this gives the
                # average loss across ALL cores, rather than their sum.
                loss_batch = loss_batch / strategy.num_replicas_in_sync
            gradients = tape.gradient(loss_batch, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            return loss_batch

        losses_per_replica = strategy.run(train_step, args=(batch_in,))
        return strategy.reduce(tf.distribute.ReduceOp.SUM,
                               losses_per_replica, axis=None)

    loss = []
    for epoch in range(epochs):
        loss_running = 0.0
        pbar = tqdm(dataset, total=steps_prx,
                    desc=f'SimCLR Training: {epoch + 1:03d}/{epochs}',
                    ncols=125, leave=True)

        for batch_num, batch in enumerate(pbar, 1):
            loss_batch = distributed_train_step(batch)
            loss_running += float(loss_batch.numpy())
            if verbose:
                pbar.set_postfix(loss=f'{loss_running / batch_num:.6f}')

        loss.append(loss_running / batch_num)
        if epoch >= patience and loss[-1] > min(loss[:-patience]):
            print('Early stopping due to no improvement in loss.')
            break

    return model, loss


# SimCLR's contrastive loss, also called NT-Xent.
@tf.function
def fun_simclr_loss(z_estimate):
    # z_estimate contains the learned representation of each image view.
    # Converting to float32 keeps the similarities numerically stable.
    z_estimate = tf.math.l2_normalize(tf.cast(z_estimate, tf.float32), axis=1)
    num = tf.shape(z_estimate)[0]  # Twice the number of original images on this core.
    temperature = 0.1

    # One score for every possible pair in this core's portion of the batch.
    similarity = tf.matmul(z_estimate, z_estimate, transpose_b=True)
    logits = similarity / temperature

    # An image should NOT be matched with itself. Give those scores a very
    # negative value so softmax ignores them in its comparisons.
    logits = logits - tf.eye(num, dtype=tf.float32) * 1e9

    # The views are interleaved: 0 pairs with 1, 2 pairs with 3, and so on.
    indices = tf.range(num)
    positive = tf.where(indices % 2 == 0, indices + 1, indices - 1)

    # For each view, reward its matching augmented view and compare it with
    # every other image view on this core. No class labels are involved.
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=positive, logits=logits)
    return tf.reduce_mean(losses)

# Function to compile a model with specified learning rate
def fun_model_compile(model, learning_rate, loss = 'categorical_crossentropy', metrics = ['accuracy']):
    # Compiles the model with Adam optimizer, categorical crossentropy as loss, and tracks accuracy
    with strategy.scope():
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                      loss=loss,
                      metrics=metrics)
    return model

# Function to print the summary of the model
def fun_model_summary(model):
    model.summary()  # Prints the summary of the model
    print('\n')  # Prints a newline for readability

# Function to define callbacks for training
def fun_model_callbacks():
    # Early stopping callback to stop training when val_loss doesn't improve
    cb_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    # Reduce learning rate callback when val_loss plateaus
    cb_reduce_lr      = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001)
    return [cb_early_stopping, cb_reduce_lr]

# Function to train the models other than SimCLR with given dataset, epochs, and callbacks
def fun_model_train(model, dataset_tr, dataset_vl, epochs, callbacks):
    model.fit(dataset_tr,  # Training data
              epochs          = epochs,  # Number of epochs to train for
              verbose         = 1,  # Verbosity mode
              shuffle         = True,  # Whether to shuffle the training data
              validation_data = dataset_vl,  # Validation data
              callbacks       = callbacks)  # Callbacks for training
    return model

# Function to evaluate the model with a given dataset
def fun_model_evaluate(model, data):
    _, acc = model.evaluate(data, verbose=0)
    return [acc]

# Function to compare the weights of base and trained models in fsp and prx tasks
def fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx):
    print('\n')
    # Print examples of weights from the 3rd layer of fully supervised base model
    print('Example of weights in the 3rd  layer of model_base_fsp: ', model_base_fsp.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the 3rd layer of pretext base model
    print('Example of weights in the 3rd  layer of model_base_prx: ', model_base_prx.layers[2].weights[0][0][0][0][:2])
    # Print examples of weights from the last layer of fully supervised trained model
    print('Example of weights in the last layer of model_fsp     : ', model_fsp.layers[-1].weights[0][:1][0][:2])
    # Print examples of weights from the last layer of pretext trained model
    print('Example of weights in the last layer of model_prx     : ', model_prx.layers[-1].weights[0][:1][0][:2])


In [12]:
#@title Training Models
# Capture the start time
t0 = time.time()

# Initiate a result dictionary
results = {}

'''
Initialize transfer learning for model_fsp.
'''

print('\nInitializing transfer learning for model_fsp\n')

# Freeze the base model's layers to prevent updates during the first phase of training.
model_base_fsp.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_fsp.layers[-2].trainable = False

# Compile the model with specified learning rate for transfer learning phase.
model_fsp = fun_model_compile(model_fsp, lr_fsp_trf)

# Store the initial parameters of the last layer of model_fsp for comparison later.
outputs_fsp_initial_parameters = model_fsp.layers[-1].get_weights()

# Display the model's structure.
fun_model_summary(model_fsp)

# Prepare callbacks for early stopping and learning rate adjustment.
callbacks = fun_model_callbacks()

# Start training the model with specified datasets and number of epochs.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_vl, epoch_fsp_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_trf'] = fun_model_evaluate(model_fsp, dataset_te)

# Check and display changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Switch to fine-tuning phase for model_fsp.
'''

print('\nStarting fine-tuning for model_fsp\n')

# Unfreeze the base model's layers for fine-tuning.
model_base_fsp.trainable = True

# Unfreeze the batch norm layer.
model_fsp.layers[-2].trainable = True

# Recompile the model with a new learning rate for fine-tuning.
model_fsp = fun_model_compile(model_fsp, lr_fsp_fnt)

# Display updated model structure.
fun_model_summary(model_fsp)

# Reinitialize callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed with fine-tuning training.
model_fsp = fun_model_train(model_fsp, dataset_tr_fsp, dataset_vl, epoch_fsp_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['fsp_fnt'] = fun_model_evaluate(model_fsp, dataset_te)

# Monitor changes in model parameters post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Initialize transfer learning for model_prx.
'''

print('\nInitializing transfer learning for model_prx\n')

# Freeze the base model's layers for transfer learning phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_prx.layers[-2].trainable = False

# Show the model's structure.
fun_model_summary(model_prx)

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                distributed_dataset_tr_prx,
                                epochs     = epoch_prx_trf,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx_trf)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Switch to fine-tuning phase for model_prx.
'''

print('\nStarting fine-tuning for model_prx\n')

# Unfreeze the base model's layers for fine-tuning adjustments.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_prx.layers[-2].trainable = True

# Begin training with specified configurations.
model_prx, _ = fun_train_simclr(model_prx,
                                distributed_dataset_tr_prx,
                                epochs     = epoch_prx_fnt,
                                verbose    = 1,
                                patience   = 1,
                                learning_rate = lr_prx_fnt)

# Evaluate changes in parameters after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Construct and prepare model_dwm for transfer learning.
'''
# Constructing model_dwm utilizing the pre-trained model_base_prx, incorporating
# a dense layer similar to that in model_fsp.
with strategy.scope():
    inputs_dwm = tf.keras.Input(shape=(res, res, 3))
    x_dwm = model_base_prx(inputs_dwm)  # Use the trained base model.
    x_dwm = tf.keras.layers.Flatten()(x_dwm)
    x_dwm = tf.keras.layers.BatchNormalization()(x_dwm)
    outputs_dwm = tf.keras.layers.Dense(10, activation='softmax')(x_dwm)  # Set for 10 output classes.
    model_dwm = tf.keras.Model(inputs_dwm, outputs_dwm)


# For equitable comparison between fsp and dwm, we align the parameters of dwm's
# final dense layer with outputs_fsp_initial_parameters.
# Additionally, we transfer and freeze the pre-trained weights of the preceding
# dense layer from prx into dwm's structure to leverage their learned
# representations.
# Set the initial parameters for fair comparison and incorporate pre-trained layers.
model_dwm.layers[-1].set_weights(outputs_fsp_initial_parameters)  # Match initial output layer parameters.
model_dwm.layers[-2].set_weights(model_prx.layers[-2].get_weights())  # Use trained batch-norm weights.

'''
Transfer learning for model_dwm.
'''

print('\nInitializing transfer learning for model_dwm\n')

# Freeze base and selected layers for the initial phase.
model_base_prx.trainable = False

# Freeze the batch norm layer to have its initial parameters (zeros and ones) intact.
model_dwm.layers[-2].trainable = False

# Compile the model for this phase.
model_dwm = fun_model_compile(model_dwm, lr_dwm_trf)

# Display the model structure.
fun_model_summary(model_dwm)

# Set callbacks for training.
callbacks = fun_model_callbacks()

# Begin training the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_vl, epoch_dwm_trf, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_trf'] = fun_model_evaluate(model_dwm, dataset_te)

# Check parameter changes after training.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)



'''
Fine-tuning for model_dwm.
'''

print('\nStarting fine-tuning for model_dwm\n')

# Unfreeze layers for fine-tuning.
model_base_prx.trainable = True

# Unfreeze the batch norm layer.
model_dwm.layers[-2].trainable = True

# Recompile with adjusted learning rate for fine-tuning.
model_dwm = fun_model_compile(model_dwm, lr_dwm_fnt)

# Display the model's updated structure.
fun_model_summary(model_dwm)

# Prepare callbacks for this phase.
callbacks = fun_model_callbacks()

# Proceed to fine-tune the model.
model_dwm = fun_model_train(model_dwm, dataset_tr_dwm, dataset_vl, epoch_dwm_fnt, callbacks)

# Evaluate the model with a specified dataset.
results ['dwm_fnt'] = fun_model_evaluate(model_dwm, dataset_te)

# Evaluate parameter changes post fine-tuning.
fun_control(model_base_fsp, model_base_prx, model_fsp, model_prx)

# Display the results.
print('---- Accuracies ----')
print(pd.DataFrame(results).head())
print('\n')
print("Time duration of experiment (sec): ", time.time() - t0)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
# from google.colab import runtime
# import time
# time.sleep(10)
# runtime.unassign()



Initializing transfer learning for model_fsp



Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16384)          │        65,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │       163,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,266,890 (27.72 MB)

 Trainable params: 163,850 (640.04 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16384)          │        65,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │       163,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,266,890 (27.72 MB)

 Trainable params: 7,150,474 (27.28 MB)

 Non-trainable params: 116,416 (454.75 KB)



Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9230 - loss: 1.0972 - val_accuracy: 0.6538 - val_loss: 9.2274 - learning_rate: 1.0000e-04
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.9721 - loss: 0.2214 - val_accuracy: 0.6731 - val_loss: 8.8997 - learning_rate: 1.0000e-04
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.9855 - loss: 0.0998 - val_accuracy: 0.6635 - val_loss: 8.9881 - learning_rate: 1.0000e-04
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9855 - loss: 0.0824 - val_accuracy: 0.6827 - val_loss: 7.5447 - learning_rate: 1.0000e-04
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9844 - loss: 0.1402 - val_accuracy: 0.7019 - val_loss: 6.6148 - learning_rate: 1.0000e-04
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9844 - loss: 0.1883 - val_accuracy: 0.6923 - val_loss: 6.7236 - learning_rate: 1.0000e-04
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.991

### Interpreting the output

The displayed table contains **test accuracy** for the baseline and the downstream classifier at their different training phases. Accuracy is the fraction of test images assigned the right class. Compare `fsp` with `dwm` *within the same experiment*; the difference estimates whether unlabeled-image pretraining helped under that experiment's settings. Early stopping can make the actual number of training epochs smaller than the maximum listed above. A small validation set makes these demonstration results variable across runs. This notebook supplies no precomputed accuracy or TPU timing: the numbers appear when you run it on your Colab TPU.

The contrastive loss compares views **within each TPU core**. It does not compare a view with negatives assigned to another core; our per-core PRX batch sizes preserve the original number of comparisons. The 104 validation images are used to select the labeled model's stopping point, and the separate test images are not used to make that choice.


# <font color="#418FDE" size="10" uppercase>**C: SimCLR Contrastive Experiments**</font>
----

In this lecture, you learned to:
* Create SimCLR experiments without pre-trained weights.
* Create SimCLR experiments with pre-trained weights.

In the next Module (Module 5), we will go over "Generative AI, Part 1."